# Capítulo 6 — Outros Aspectos de Regressão

**Livro:** começa na página **137** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
set.seed(42)
options(digits = 4, repr.plot.width = 7, repr.plot.height = 4.5)
suppressPackageStartupMessages({ library(glmnet); library(rpart); library(randomForest); library(FNN); library(e1071) })
cat("Pronto.", R.version.string, "
")

### PDP e ICE · PDF p. 140

#### 🧪 Exercício: Gráfico de dependência parcial à mão

Com uma floresta em `na.omit(airquality)`, calcule o PDP de `Temp`: para cada valor t de uma grade, troque `Temp` por t em **todas** as linhas, preveja e tire a média. Desenhe a curva.

In [ ]:
aq <- na.omit(airquality)
rf <- randomForest(Ozone ~ ., aq)
grade <- seq(min(aq$Temp), max(aq$Temp), length.out = 30)
pdp <- sapply(grade, function(t) { d <- aq; d$Temp <- t; mean(predict(rf, d)) })
plot(grade, pdp, type = "l", lwd = 2, xlab = "Temp", ylab = "Ozone previsto (média)")

## Inferência Conformal · PDF p. 145

#### 🧪 Exercício: Intervalos de predição conformais

Conformal *split*: ajuste uma regressão em metade dos dados do seno, calcule os resíduos absolutos na outra metade e use o quantil de 90% como meia-largura do intervalo. Confira a cobertura em 5.000 pontos novos.

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
d <- sim_seno(400); te <- sim_seno(5000)
i <- sample(400, 200)
m <- lm(y ~ poly(x, 5), d[i, ])
res <- abs(d$y[-i] - predict(m, d[-i, ]))
q <- quantile(res, 0.9)
mean(abs(te$y - predict(m, te)) <= q)  # deve ficar perto de 0,90